# HW2 : Convergence uniforme et limites de l'apprentissage

**Université de Haute-Alsace – M2 IM / IMDS / MIAGE – Statistical and Machine Learning**

Ce notebook simule directement les distributions de probabilité qui apparaissent dans les résultats théoriques du cours :

* **Partie I** : fluctuations du risque empirique autour du risque vrai, borne de Hoeffding, convergence uniforme sur une classe finie, complexité d'échantillonnage ;
* **Partie II** : *No-Free-Lunch* avec des fonctions cibles totalement aléatoires ;
* **Partie III** : introduction d'un biais inductif (seuils) et comparaison avec la Partie II.

Seuls `numpy` et `matplotlib` sont utilisés. Une graine fixe (`default_rng(42)`) rend les résultats reproductibles **si les cellules sont exécutées dans l'ordre**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
TOL = 1e-12   # tolérance numérique pour les comparaisons "> epsilon" sur des flottants

---
# Partie I – Simulation de la convergence uniforme

On considère $\mathcal H=\{h_1,\dots,h_K\}$ où chaque $h_j$ a un risque vrai $L_D(h_j)=p_j$. Sur un échantillon de taille $m$, le nombre d'erreurs vaut $E_j\sim\text{Binomial}(m,p_j)$ et $L_S(h_j)=E_j/m$.

## 1. Le risque empirique comme variable aléatoire

**Théorie** : $\mathbb E[L_S(h_j)]=p_j$ (estimateur sans biais) et $\operatorname{Var}(L_S(h_j))=p_j(1-p_j)/m$, donc l'écart-type décroît en $1/\sqrt m$.

**Choix des paramètres** : $p\in\{0.1,0.5,0.8\}$ (faible, maximale et forte variance), $m\in\{10,50,200,1000\}$ (de très petit à grand échantillon), $20\,000$ répétitions Monte-Carlo (erreur Monte-Carlo sur la moyenne $\lesssim 0.004$).

In [ ]:
p_values_1 = [0.1, 0.5, 0.8]
m_values_1 = [10, 50, 200, 1000]
n_rep = 20_000

fig, axes = plt.subplots(len(p_values_1), len(m_values_1), figsize=(15, 9), sharex=True)
rows = []
for i, p in enumerate(p_values_1):
    for j, m in enumerate(m_values_1):
        E = rng.binomial(m, p, size=n_rep)          # (a) simulation de E ~ Bin(m, p)
        LS = E / m                                  # (b) risque empirique
        rows.append((p, m, LS.mean(), LS.std(), np.sqrt(p * (1 - p) / m)))
        ax = axes[i, j]                             # (c) visualisation
        ax.hist(LS, bins=np.linspace(0, 1, 41), density=True, alpha=0.75)
        ax.axvline(p, color="r", ls="--", label="$L_D(h)=p$")
        ax.axvline(LS.mean(), color="k", ls=":", label="moyenne empirique")
        ax.set_title(f"p={p}, m={m}", fontsize=10)
        if i == 0 and j == 0:
            ax.legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("$L_S(h)$")
plt.suptitle("Distribution de $L_S(h)$ pour différents $p$ et $m$", y=1.00)
plt.tight_layout(); plt.show()

# (d) comparaison moyenne empirique / risque vrai, écart-type empirique / théorique
print(f"{'p':>5} {'m':>6} {'moyenne emp.':>14} {'écart-type emp.':>16} {'sqrt(p(1-p)/m)':>16}")
for p, m, mu, sd, sd_th in rows:
    print(f"{p:5.2f} {m:6d} {mu:14.4f} {sd:16.4f} {sd_th:16.4f}")

**Interprétation.** La moyenne empirique de $L_S(h)$ coïncide avec $p$ à l'erreur Monte-Carlo près : le risque empirique est un estimateur **non biaisé** du risque vrai. L'écart-type empirique colle à $\sqrt{p(1-p)/m}$ et diminue d'un facteur $\approx 10$ quand $m$ est multiplié par 100 : la distribution se resserre autour de $p$ (loi des grands nombres) et devient de plus en plus gaussienne (TCL). À $m=10$, $L_S(h)$ peut être très éloigné de $p$ et prend peu de valeurs distinctes ; à $m=1000$ il est quasi déterministe. La variance est maximale pour $p=0.5$.

## 2. Écart entre risque empirique et risque vrai (borne de Hoeffding)

On estime $\Pr[|L_S(h)-L_D(h)|>\varepsilon]$ par la fréquence des écarts $>\varepsilon$ sur $20\,000$ échantillons, et on la compare à $2e^{-2m\varepsilon^2}$. On prend $\varepsilon=0.1$, $m$ de 5 à 600 par pas de 5, et $p\in\{0.1,0.5,0.9\}$ ($p=0.5$ est le cas le plus défavorable ; $p=0.9$ est symétrique de $p=0.1$, ce que l'on vérifie).

In [ ]:
def hoeffding(m, eps):
    return 2 * np.exp(-2 * np.asarray(m) * eps ** 2)

eps = 0.1
n_rep = 20_000
m_grid2 = np.arange(5, 601, 5)
p_values_2 = [0.1, 0.5, 0.9]

res2 = {p: np.array([np.mean(np.abs(rng.binomial(m, p, size=n_rep) / m - p) > eps + TOL)
                     for m in m_grid2]) for p in p_values_2}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, logy in zip(axes, [False, True]):
    for p in p_values_2:
        y = res2[p]
        ax.plot(m_grid2, np.where(y > 0, y, np.nan) if logy else y, label=f"expérience, p={p}")
    ax.plot(m_grid2, np.minimum(1, hoeffding(m_grid2, eps)), "k--", lw=2, label="borne de Hoeffding (tronquée à 1)")
    ax.set_xlabel("m"); ax.set_ylabel(r"$\Pr[|L_S(h)-L_D(h)|>\varepsilon]$")
    if logy:
        ax.set_yscale("log"); ax.set_ylim(1e-5, 2); ax.set_title("échelle logarithmique")
    else:
        ax.set_title(f"échelle linéaire (ε={eps})")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Rapport borne / probabilité expérimentale (p=0.5) :")
for m in [50, 100, 200, 300]:
    k = np.where(m_grid2 == m)[0][0]
    print(f"  m={m:4d}  expérience={res2[0.5][k]:.5f}  Hoeffding={hoeffding(m, eps):.5f}")

**Interprétation.** Les probabilités expérimentales décroissent exponentiellement avec $m$ (droite en échelle log), comme la borne, et restent **toujours en dessous** de celle-ci. La borne est donc **valide mais conservatrice** : elle est valable pour *toute* loi de $[0,1]$ et n'exploite pas la variance $p(1-p)$ ; pour $p=0.1$ l'écart est énorme, car la variance est petite. Même pour $p=0.5$ (cas le pire) la borne surestime nettement, notamment à cause du facteur 2 (bornes unilatérales sommées) et parce que Hoeffding ne capture pas la vraie vitesse gaussienne $\approx e^{-m\varepsilon^2/(2p(1-p))}$. Les courbes en dents de scie viennent de la discrétisation de $E/m$.

## 3. De une hypothèse à la convergence uniforme

Pour chaque échantillon simulé, on tire $E_j\sim\text{Bin}(m,p_j)$ pour **tous** les $j$ et on calcule $\Delta_S=\max_j|L_S(h_j)-p_j|$. On compare $\Pr[\Delta_S>\varepsilon]$ à $\min(1,\,2|H|e^{-2m\varepsilon^2})$.

Choix : $K=50$ hypothèses avec $p_j$ régulièrement répartis dans $[0.05,0.95]$, $\varepsilon=0.1$, $m\in[10,1000]$, $5000$ répétitions.

In [ ]:
def delta_samples(m, p_vec, n_rep):
    """n_rep réalisations de Delta_S = max_j |E_j/m - p_j|, avec E_j ~ Bin(m, p_j) indépendantes."""
    E = rng.binomial(m, p_vec, size=(n_rep, len(p_vec)))
    return np.abs(E / m - p_vec).max(axis=1)

def make_p(K):
    return np.linspace(0.05, 0.95, K)

K, eps, n_rep = 50, 0.1, 5000
m_grid3 = np.arange(10, 1001, 20)
p_vec = make_p(K)

prob_unif = np.array([np.mean(delta_samples(m, p_vec, n_rep) > eps + TOL) for m in m_grid3])
prob_single = np.array([np.mean(np.abs(rng.binomial(m, 0.5, size=n_rep) / m - 0.5) > eps + TOL) for m in m_grid3])
bound_unif = np.minimum(1, 2 * K * np.exp(-2 * m_grid3 * eps ** 2))
bound_single = np.minimum(1, hoeffding(m_grid3, eps))

plt.figure(figsize=(8, 5))
plt.plot(m_grid3, prob_unif, "o-", ms=3, label=r"expérience : $\Pr[\Delta_S>\varepsilon]$ (K=50)")
plt.plot(m_grid3, bound_unif, "r--", label=r"borne $2|H|e^{-2m\varepsilon^2}$")
plt.plot(m_grid3, prob_single, "g-", alpha=0.7, label="une seule hypothèse (p=0.5)")
plt.plot(m_grid3, bound_single, "g--", alpha=0.7, label="Hoeffding, une hypothèse")
plt.xlabel("m"); plt.ylabel("probabilité"); plt.title(f"Convergence uniforme, ε={eps}")
plt.legend(); plt.show()

m0 = m_grid3[np.argmax(bound_unif < 1)]
print(f"La borne uniforme devient non triviale (<1) à partir de m ≈ {m0}")
print(f"Probabilité expérimentale à ce m : {prob_unif[np.argmax(bound_unif<1)]:.4f}")

**Interprétation.** $\Pr[\Delta_S>\varepsilon]$ est toujours **plus grande** que la probabilité pour une seule hypothèse : il faut qu'*aucune* des $K$ hypothèses ne s'écarte, donc le « mauvais événement » est une union d'événements (d'où la borne de l'union, facteur $|H|$). Pour $m$ petit la probabilité est proche de 1 : il y a presque toujours *au moins une* hypothèse dont le risque empirique est trompeur. La borne théorique est triviale ($\ge 1$) jusqu'à $m\approx\log(2|H|)/(2\varepsilon^2)$, puis décroît exponentiellement ; elle est conservatrice car l'inégalité de l'union suppose le pire cas (événements disjoints), alors qu'ils se chevauchent, et chaque terme est lui-même surestimé par Hoeffding.

## 4. Effet de la taille de la classe $|H|$

On fixe $\varepsilon=0.1$ et on trace $\Pr[\Delta_S>\varepsilon]$ pour $|H|\in\{5,10,50,100,500\}$ (traits pleins : expérience ; pointillés : borne).

In [ ]:
K_list = [5, 10, 50, 100, 500]
eps, n_rep = 0.1, 2000
m_grid4 = np.arange(10, 1201, 20)

res4 = {K: np.array([np.mean(delta_samples(m, make_p(K), n_rep) > eps + TOL) for m in m_grid4]) for K in K_list}

plt.figure(figsize=(9, 5.5))
for K, c in zip(K_list, plt.cm.viridis(np.linspace(0, 0.9, len(K_list)))):
    plt.plot(m_grid4, res4[K], color=c, lw=2, label=f"|H|={K}")
    plt.plot(m_grid4, np.minimum(1, 2 * K * np.exp(-2 * m_grid4 * eps ** 2)), color=c, ls="--", alpha=0.7)
plt.xlabel("m"); plt.ylabel(r"$\Pr[\Delta_S>\varepsilon]$")
plt.title(f"Effet de |H| (ε={eps}) – trait plein : expérience, pointillés : borne")
plt.legend(); plt.show()

# décalage de la courbe : m nécessaire pour atteindre une proba de 0.05, en fonction de log|H|
for K in K_list:
    idx = np.argmax(res4[K] <= 0.05)
    print(f"|H|={K:4d}  log|H|={np.log(K):5.2f}  m expérimental (Pr<=0.05) ≈ {m_grid4[idx]}")

**Interprétation.**
* **Effet de $|H|$** : à $m$ fixé, plus la classe est grande, plus $\Pr[\Delta_S>\varepsilon]$ est élevée : la convergence uniforme est plus difficile à obtenir car il y a davantage d'hypothèses susceptibles d'être « trompeuses ».
* **Besoin de données** : oui, une classe plus grande demande plus de données pour la même garantie $(\varepsilon,\delta)$.
* **Lien avec $\log|H|$** : les courbes sont décalées vers la droite de façon **régulière** quand $|H|$ est multiplié par un facteur constant (5→50→500), pas proportionnellement à $|H|$. Les $m$ expérimentaux croissent à peu près linéairement en $\log|H|$, ce qui est exactement la dépendance $m\gtrsim\log(2|H|/\delta)/(2\varepsilon^2)$ : on peut multiplier la taille de la classe par 10 au prix d'un surcoût **additif** (≈ $\log 10/(2\varepsilon^2)$ exemples).

## 5. Complexité d'échantillonnage expérimentale

On fixe $\delta=0.05$ et deux niveaux de précision $\varepsilon\in\{0.1,0.2\}$. On cherche le plus petit $m$ de la grille testée tel que $\Pr[\Delta_S>\varepsilon]\le\delta$, et on le compare à $m_{\text{th}}=\left\lceil\dfrac{\log(2|H|/\delta)}{2\varepsilon^2}\right\rceil$.

*Remarque* : la probabilité est estimée par Monte-Carlo (écart-type $\approx 0.005$ pour $\delta=0.05$ avec 2000 répétitions) ; le $m$ expérimental est donc connu à une petite incertitude près.

In [ ]:
delta, n_rep = 0.05, 2000
configs = {0.1: np.arange(5, 701, 5), 0.2: np.arange(2, 201, 2)}   # grille de m par epsilon
rows5, curves5 = [], {}
for eps5, grid in configs.items():
    for K in K_list:
        pv = make_p(K)
        pr = np.array([np.mean(delta_samples(m, pv, n_rep) > eps5 + TOL) for m in grid])
        m_exp = grid[np.argmax(pr <= delta)] if (pr <= delta).any() else None
        m_th = int(np.ceil(np.log(2 * K / delta) / (2 * eps5 ** 2)))
        rows5.append((eps5, K, m_exp, m_th)); curves5[(eps5, K)] = (grid, pr)

print(f"{'eps':>5} {'|H|':>5} {'m expérimental':>15} {'m théorique':>12} {'ratio th/exp':>13}")
for e, K, me, mt in rows5:
    print(f"{e:5.2f} {K:5d} {me:15d} {mt:12d} {mt/me:13.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, eps5 in zip(axes, configs):
    for K, c in zip(K_list, plt.cm.viridis(np.linspace(0, 0.9, len(K_list)))):
        g, pr = curves5[(eps5, K)]
        ax.plot(g, pr, color=c, label=f"|H|={K}")
        mt = [r[3] for r in rows5 if r[0] == eps5 and r[1] == K][0]
        ax.axvline(mt, color=c, ls=":", alpha=0.8)
    ax.axhline(delta, color="k", ls="--", label=r"$\delta$")
    ax.set_title(f"ε={eps5}  (pointillés verticaux : $m_{{th}}$)")
    ax.set_xlabel("m"); ax.set_ylabel(r"$\Pr[\Delta_S>\varepsilon]$"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Interprétation.** Le $m$ théorique est **toujours supérieur** au $m$ expérimental (ici d'un facteur environ 1,5 à 2,2, le plus grand pour les petites classes) : la borne est une **condition suffisante** (si $m\ge m_{\rm th}$ alors $\Pr[\Delta_S>\varepsilon]\le\delta$ est *garanti* pour **toute** loi et toute valeur de $p_j$), pas une condition nécessaire. Elle est obtenue par Hoeffding + borne de l'union, deux majorations pessimistes ; or ici les $p_j$ sont fixés (plusieurs sont loin de 0.5, donc de faible variance) et les erreurs des $h_j$ sont indépendantes. Le $m$ expérimental, lui, est valable pour *cette* configuration seulement ; la borne est la bonne quantité si l'on veut une garantie sans hypothèse sur la distribution. Les deux croissent néanmoins de la même façon en $\log|H|$ et en $1/\varepsilon^2$.

---
# Partie II – Simulation du No-Free-Lunch

Domaine $\mathcal X=\{1,\dots,N\}$ muni de la loi uniforme. À chaque répétition, on tire une **nouvelle cible** $f(x)\sim\text{Bernoulli}(1/2)$ indépendamment pour chaque $x$, puis un échantillon $S\sim D^m$ (étiquetés par $f$). L'algorithme mémorise : il prédit l'étiquette observée pour un point vu, et une règle par défaut (initialement $0$) pour un point non vu.

*Implémentation* : les points sont indexés de $0$ à $N-1$ dans le code (équivalent à $1,\dots,N$). Tout est vectorisé sur les répétitions.

In [ ]:
def sim_nfl(N, m, n_rep, strategy="zero", q=0.5):
    """Simule n_rep tâches : cible f(x)~Bernoulli(q) (q=1/2 : totalement aléatoire), échantillon de m points uniformes.
    Apprenant par mémorisation ; règle sur les points non vus : 'zero', 'one', 'random' ou 'majority'.
    Retourne la fraction de points non vus, le risque vrai sur tout X, et le risque restreint aux points non vus."""
    f = (rng.random((n_rep, N)) < q).astype(int)
    S = rng.integers(0, N, size=(n_rep, m))
    seen = np.zeros((n_rep, N), dtype=bool)
    if m > 0:
        seen[np.arange(n_rep)[:, None], S] = True
    n_seen = seen.sum(1)
    if strategy == "zero":
        default = 0
    elif strategy == "one":
        default = 1
    elif strategy == "random":
        default = rng.integers(0, 2, size=(n_rep, N))
    elif strategy == "majority":     # étiquette majoritaire parmi les points vus (0 en cas d'égalité / aucun point)
        default = ((f * seen).sum(1) * 2 > n_seen).astype(int)[:, None]
    else:
        raise ValueError(strategy)
    pred = np.where(seen, f, default)                  # étiquette observée = f(x) (données sans bruit)
    err = pred != f
    n_unseen = N - n_seen
    return dict(unseen=n_unseen / N,
                risk=err.mean(1),
                risk_unseen=np.where(n_unseen > 0, err.sum(1) / np.maximum(n_unseen, 1), np.nan))

# petit test de cohérence
r = sim_nfl(10, 5, 3)
print({k: np.round(v, 2) for k, v in r.items()})

## 6. Points vus et non vus

Pour $x$ fixé, chaque tirage évite $x$ avec probabilité $1-\frac1N$ ; les $m$ tirages étant indépendants,
$$\Pr[x\text{ non vu}]=\left(1-\tfrac1N\right)^m .$$
Soit $U=\frac1N\sum_{x}\mathbf 1[x\text{ non vu}]$ la fraction de points non vus. Par **linéarité de l'espérance** (aucune indépendance entre les $x$ n'est requise),
$$\mathbb E[U]=\frac1N\sum_x\Pr[x\text{ non vu}]=\left(1-\tfrac1N\right)^m .$$

Choix : $N\in\{20,50,100\}$, $m$ de 1 à $4N$ (soit jusqu'à 4 fois la taille du domaine), 4000 répétitions.

In [ ]:
N_list = [20, 50, 100]
n_rep = 4000
fig, axes = plt.subplots(1, len(N_list), figsize=(15, 4.2))
for ax, N in zip(axes, N_list):
    m_vals = np.unique(np.linspace(1, 4 * N, 25).astype(int))
    emp = np.array([sim_nfl(N, m, n_rep)["unseen"].mean() for m in m_vals])
    th = (1 - 1 / N) ** m_vals
    ax.plot(m_vals, th, "r-", label=r"théorie $(1-1/N)^m$")
    ax.plot(m_vals, emp, "ko", ms=4, label="simulation")
    ax.set_title(f"N={N}"); ax.set_xlabel("m"); ax.set_ylabel("fraction de points non vus"); ax.legend()
    print(f"N={N:4d}  écart absolu max théorie/simulation = {np.abs(emp - th).max():.4f}")
plt.tight_layout(); plt.show()

**Interprétation.** La simulation reproduit la formule à l'erreur Monte-Carlo près. La fraction de points non vus décroît exponentiellement ($\approx e^{-m/N}$) mais reste **loin de zéro** tant que $m\lesssim N$ : par exemple pour $m=N$, environ $e^{-1}\approx 37\%$ du domaine n'est jamais observé. L'espérance de la *fraction* est égale à la probabilité d'un point fixe par linéarité de l'espérance.

## 7. Erreur sur les points non vus

Avec la règle « prédire 0 », l'apprenant ne se trompe jamais sur un point vu (il connaît $f(x)$) et se trompe sur un point non vu $x$ ssi $f(x)=1$. Les $f(x)$ étant i.i.d. Bernoulli(1/2) et **indépendants de l'échantillon** (la cible est tirée sans lien avec les $x_i$ observés) :
$$\mathbb E[L_D(A(S))]=\frac1N\sum_x \Pr[x\text{ non vu}]\cdot\Pr[f(x)=1]=\frac12\left(1-\tfrac1N\right)^m .$$
On compare cette formule à la simulation, ainsi qu'au risque mesuré **uniquement sur les points non vus** (théoriquement $1/2$).

In [ ]:
n_rep = 4000
fig, axes = plt.subplots(1, len(N_list), figsize=(15, 4.2))
for ax, N in zip(axes, N_list):
    m_vals = np.unique(np.linspace(1, 4 * N, 25).astype(int))
    res = [sim_nfl(N, m, n_rep, "zero") for m in m_vals]
    emp = np.array([r["risk"].mean() for r in res])
    se = np.array([r["risk"].std() / np.sqrt(n_rep) for r in res])
    emp_un = np.array([np.nanmean(r["risk_unseen"]) for r in res])
    th = 0.5 * (1 - 1 / N) ** m_vals
    ax.plot(m_vals, th, "r-", label=r"théorie $\frac{1}{2}(1-1/N)^m$")
    ax.errorbar(m_vals, emp, yerr=2 * se, fmt="ko", ms=3, label="simulation (±2 s.e.)")
    ax.plot(m_vals, emp_un, "b^", ms=3, alpha=0.6, label="risque sur points non vus")
    ax.axhline(0.5, color="gray", ls=":")
    ax.set_title(f"N={N}"); ax.set_xlabel("m"); ax.set_ylabel(r"$\mathbb{E}[L_D(A(S))]$"); ax.legend(fontsize=8)
    print(f"N={N:4d}  écart max |sim - théorie| = {np.abs(emp - th).max():.4f}   "
          f"risque moyen sur points non vus ≈ {np.nanmean(emp_un):.3f}")
plt.tight_layout(); plt.show()

**Interprétation.** La simulation confirme $\mathbb E[L_D(A(S))]=\frac12(1-1/N)^m$ : le risque est exactement la moitié de la fraction de points non vus. Il décroît uniquement *parce que* de plus en plus de points sont mémorisés, et non parce que l'algorithme **généralise** : l'erreur sur les points non vus reste égale à $1/2$ (courbe bleue), c'est-à-dire le hasard, quel que soit $m$.

## 8. Autres règles pour les points non vus

On compare quatre règles sur les points non vus : toujours $0$, toujours $1$, Bernoulli(1/2), et (en plus) l'étiquette **majoritaire** des points vus. Cibles tirées uniformément ($q=1/2$), puis, pour illustrer le NFL, cibles **biaisées** ($f(x)\sim\text{Bernoulli}(q)$ avec $q\ne1/2$).

In [ ]:
strategies = ["zero", "one", "random", "majority"]
N, n_rep = 50, 6000
m_vals = np.unique(np.linspace(1, 4 * N, 20).astype(int))

plt.figure(figsize=(8.5, 5))
for s, mk in zip(strategies, ["o", "s", "^", "x"]):
    emp = [sim_nfl(N, m, n_rep, s)["risk"].mean() for m in m_vals]
    plt.plot(m_vals, emp, mk, ms=5, label=s)
plt.plot(m_vals, 0.5 * (1 - 1 / N) ** m_vals, "k-", label=r"théorie $\frac{1}{2}(1-1/N)^m$")
plt.xlabel("m"); plt.ylabel("risque vrai moyen"); plt.title(f"Cibles totalement aléatoires (N={N})")
plt.legend(); plt.show()

print(f"Risque moyen à m=N={N} (erreur standard ≈ 0.001) :")
for s in strategies:
    r = sim_nfl(N, N, 20000, s)["risk"]
    print(f"  {s:9s} {r.mean():.4f} ± {r.std()/np.sqrt(len(r)):.4f}")
print(f"  théorie    {0.5 * (1 - 1 / N) ** N:.4f}")

In [ ]:
# Si les cibles ne sont PAS uniformes (biais q), le "meilleur" choix change : pas de règle universelle.
q_list = [0.0, 0.1, 0.5, 0.9, 1.0]
N, m, n_rep = 50, 25, 6000
vals = np.array([[sim_nfl(N, m, n_rep, s, q=q)["risk"].mean() for s in strategies] for q in q_list])

x = np.arange(len(q_list)); w = 0.2
plt.figure(figsize=(9, 4.8))
for k, s in enumerate(strategies):
    plt.bar(x + (k - 1.5) * w, vals[:, k], w, label=s)
plt.xticks(x, [f"q={q}" for q in q_list]); plt.ylabel(f"risque moyen (N={N}, m={m})")
plt.title(r"Cibles biaisées : $f(x)\sim$ Bernoulli(q)"); plt.legend(); plt.show()
print("Risque moyen :"); print(f"{'q':>5} " + " ".join(f"{s:>9}" for s in strategies))
for q, row in zip(q_list, vals):
    print(f"{q:5.1f} " + " ".join(f"{v:9.4f}" for v in row))

**Interprétation.** Pour des cibles complètement aléatoires, **les quatre règles ont le même risque moyen** $\frac12(1-1/N)^m$, aux fluctuations Monte-Carlo près : aucune ne surpasse systématiquement les autres. Sur un point non vu, $f(x)$ est indépendant de tout ce que l'algorithme a vu ; toute règle (même « intelligente », comme la majorité) est donc correcte avec probabilité $1/2$.

Le second graphique est le cœur du **No-Free-Lunch** : dès que la distribution des cibles favorise certaines fonctions ($q\neq 1/2$), la règle « 0 » gagne pour $q<1/2$, la règle « 1 » pour $q>1/2$, et la règle « majorité » (qui *suppose* que les points non vus ressemblent aux points vus) est bonne dès que $q\neq1/2$. Mais chaque règle est **mauvaise** pour au moins une distribution de cibles ; moyennées sur *toutes* les cibles possibles, elles se valent. Aucun apprenant ne généralise mieux que les autres sans hypothèse a priori sur la tâche.

---
# Partie III – Introduction de connaissances a priori

Même domaine $\mathcal X=\{1,\dots,N\}$ (loi uniforme), mais la cible est maintenant un **seuil** $f_\theta(x)=\mathbf 1[x>\theta]$, $\theta$ tiré uniformément dans $\{1,\dots,N-1\}$. L'apprenant utilise la classe $\mathcal H=\{h_t:t\in\{1,\dots,N-1\}\}$ et **ERM**.

*ERM vectorisé* : l'erreur d'entraînement de $h_t$ est $\#\{i:x_i>t,\,y_i=0\}+\#\{i:x_i\le t,\,y_i=1\}$, calculée pour tous les $t$ par sommes cumulées. Les ex-æquo sont départagés **au hasard** (ERM n'impose pas de règle).

Le risque vrai de $h_t$ sur le domaine fini vaut $|t-\theta|/N$ (nombre de points entre les deux seuils, divisé par $N$) ; on le calcule néanmoins directement à partir de la définition.

In [ ]:
def sim_threshold(N, m, n_rep):
    theta = rng.integers(1, N, size=n_rep)                   # θ ∈ {1,…,N-1}
    X = rng.integers(1, N + 1, size=(n_rep, m))               # points de l'échantillon dans {1,…,N}
    Y = (X > theta[:, None]).astype(int)
    rows = np.repeat(np.arange(n_rep), m)
    flat = rows * N + (X.ravel() - 1)
    C0 = np.bincount(flat[Y.ravel() == 0], minlength=n_rep * N).reshape(n_rep, N)   # nb de points de label 0 en chaque x
    C1 = np.bincount(flat[Y.ravel() == 1], minlength=n_rep * N).reshape(n_rep, N)
    cum0, cum1 = C0.cumsum(1), C1.cumsum(1)                   # cum[:, t-1] = nb de points avec x <= t
    tot0 = cum0[:, -1:]
    err_train = (tot0 - cum0[:, :N - 1]) + cum1[:, :N - 1]    # colonne t-1 <-> hypothèse h_t, t=1..N-1
    t_hat = np.argmin(err_train + 0.5 * rng.random(err_train.shape), axis=1) + 1   # ERM, ex-æquo au hasard
    # risque vrai sur tout le domaine, d'après la définition
    xs = np.arange(1, N + 1)
    err_dom = (xs > t_hat[:, None]) != (xs > theta[:, None])
    seen = np.zeros((n_rep, N), dtype=bool); seen[np.arange(n_rep)[:, None], X - 1] = True
    n_un = N - seen.sum(1)
    risk_un = np.where(n_un > 0, (err_dom & ~seen).sum(1) / np.maximum(n_un, 1), np.nan)
    return dict(risk=err_dom.mean(1), risk_unseen=risk_un, unseen=n_un / N, train_err=err_train.min(1) / m,
                t_hat=t_hat, theta=theta)

# vérification de l'ERM vectorisé contre une version naïve
N_, m_ = 12, 6
r = sim_threshold(N_, m_, 1)
print("Erreur d'entraînement minimale (réalisable => 0) :", r["train_err"], "| risque vrai :", r["risk"], "| |t-θ|/N =", abs(r["t_hat"] - r["theta"]) / N_)
assert (sim_threshold(30, 15, 200)["train_err"] == 0).all()   # cible dans H => ERM atteint 0 erreur d'entraînement

## 9. Apprentissage avec un biais inductif adapté

On estime $\mathbb E[L_D(h_S)]$ pour plusieurs $m$ et $N\in\{20,50,100,200\}$ (3000 répétitions).

In [ ]:
N_list9 = [20, 50, 100, 200]
n_rep = 3000
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for N in N_list9:
    m_vals = np.unique(np.linspace(1, 2 * N, 25).astype(int))
    thr = np.array([sim_threshold(N, m, n_rep)["risk"].mean() for m in m_vals])
    for ax in axes:
        ax.plot(m_vals, thr, "o-", ms=3, label=f"seuils (ERM), N={N}")
axes[0].set_title("Risque de l'ERM sur seuils"); axes[1].set_title("idem, échelle log-log"); axes[1].set_xscale("log"); axes[1].set_yscale("log")
for ax in axes:
    ax.set_xlabel("m"); ax.set_ylabel(r"$\mathbb{E}[L_D(h_S)]$"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

N = 100
print(f"N={N} : comparaison à m = N/5, N/2, N")
for m in [N // 5, N // 2, N]:
    rt = sim_threshold(N, m, 5000)["risk"].mean()
    rr = sim_nfl(N, m, 5000)["risk"].mean()
    print(f"  m={m:4d}  seuils/ERM : {rt:.4f}   cibles aléatoires (mémorisation) : {rr:.4f}")

**Interprétation.** Le risque de l'ERM sur seuils s'effondre dès que $m$ est faible devant $N$ (quelques dizaines d'exemples suffisent pour $N=200$), alors que dans la Partie II il fallait voir une fraction importante du domaine pour que le risque devienne petit. La décroissance est de type $\sim1/m$ (pente $\approx -1$ en log-log) et dépend peu de $N$ : le risque est gouverné par la taille de l'intervalle entre les deux exemples encadrant $\theta$, de l'ordre de $1/m$ de la masse totale : conformément à la théorie du cas réalisable, $m\gtrsim(\log|H|+\log(1/\delta))/\varepsilon$ exemples suffisent, avec $|H|=N-1$ seuils.

## 10. Cibles aléatoires vs cibles structurées

On compare, pour $N=50$ et $N=200$ : (a) la mémorisation sur cibles totalement aléatoires (simulation + théorie) ; (b) l'ERM sur seuils avec cible seuil. En bas : le risque mesuré **uniquement sur les points non vus**, c'est-à-dire la capacité à généraliser.

In [ ]:
n_rep = 3000
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for col, N in enumerate([50, 200]):
    m_vals = np.unique(np.linspace(1, 2 * N, 25).astype(int))
    rnd = [sim_nfl(N, m, n_rep, "zero") for m in m_vals]
    thr = [sim_threshold(N, m, n_rep) for m in m_vals]
    for row, key, ylab in [(0, "risk", "risque vrai (tout le domaine)"), (1, "risk_unseen", "risque sur points NON vus")]:
        ax = axes[row, col]
        ax.plot(m_vals, [np.nanmean(r[key]) for r in rnd], "ro-", ms=3, label="(a) cibles aléatoires, mémorisation")
        ax.plot(m_vals, [np.nanmean(r[key]) for r in thr], "bs-", ms=3, label="(b) cibles seuils, ERM sur seuils")
        if row == 0:
            ax.plot(m_vals, 0.5 * (1 - 1 / N) ** m_vals, "k--", label=r"théorie (a) : $\frac{1}{2}(1-1/N)^m$")
        ax.set_title(f"N={N}"); ax.set_xlabel("m"); ax.set_ylabel(ylab); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

N, m = 100, 50
a, b = sim_nfl(N, m, 5000), sim_threshold(N, m, 5000)
print(f"N={N}, m={m} : fraction de points non vus ≈ {a['unseen'].mean():.2f} (a) et {b['unseen'].mean():.2f} (b)")
print(f"  risque sur points non vus : (a) {np.nanmean(a['risk_unseen']):.3f}   (b) {np.nanmean(b['risk_unseen']):.3f}")

**Interprétation.**
* **Cibles aléatoires** : le risque ne baisse qu'en proportion des points mémorisés ; sur les points non vus il reste à $\approx 1/2$ (graphe du bas). L'échantillon ne contient *aucune information* sur les étiquettes non observées, puisque les $f(x)$ sont indépendants.
* **Cibles seuils + classe de seuils** : le risque sur les points **non vus** est faible, bien inférieur à $1/2$, bien qu'une fraction importante du domaine n'ait jamais été vue. Chaque exemple $(x_i,y_i)$ contraint le seuil $\theta$ et donc l'étiquette de **tous les points voisins** ; il n'y a que $N-1$ fonctions possibles (au lieu de $2^N$), de sorte que quelques exemples éliminent la quasi-totalité des hypothèses.
* **Rôle de l'hypothèse « la cible est un seuil »** : c'est elle (le *biais inductif*) qui relie les points vus aux points non vus. Elle restreint $\mathcal H$ à une classe de petite taille ($\log|H|=\log(N-1)\ll N\log 2$) et réalisable : on contourne ainsi le NFL, au prix d'une garantie qui n'est valable que si la cible appartient bien à cette famille (si la cible était aléatoire, l'ERM sur seuils ne ferait pas mieux que le hasard sur les points non vus).

## 11. Conclusion générale

**Concentration du risque empirique.** $L_S(h)$ est une moyenne de $m$ variables de Bernoulli indépendantes d'espérance $L_D(h)$ : sa variance vaut $L_D(h)(1-L_D(h))/m$ et la probabilité d'un écart supérieur à $\varepsilon$ décroît exponentiellement en $m\varepsilon^2$ (Hoeffding). Les histogrammes de la Partie I se resserrent donc autour de $p$ en $1/\sqrt m$. La borne de Hoeffding est valide mais conservatrice (elle ignore la variance réelle).

**Pourquoi un contrôle simultané ?** L'ERM choisit $h_S$ *en fonction* de l'échantillon : il peut tomber sur l'hypothèse dont le risque empirique est trompeur. Garantir $L_D(h_S)\approx L_S(h_S)$ exige donc que **toutes** les hypothèses soient contrôlées en même temps, soit $\Delta_S=\max_h|L_S(h)-L_D(h)|\le\varepsilon$. Contrôler une seule hypothèse fixée à l'avance ne suffit pas.

**Rôle de $|H|$.** Cet événement est l'intersection de $|H|$ événements ; la borne de l'union donne $2|H|e^{-2m\varepsilon^2}$. À $m$ fixé, plus $|H|$ est grand, plus il y a de risque d'une hypothèse trompeuse ; le $m$ nécessaire croît en $\log|H|$ (et non en $|H|$), comme observé dans les parties 4 et 5.

**Conservatisme des bornes.** Les $m$ expérimentaux sont inférieurs aux $m$ théoriques : la borne de l'union suppose des événements disjoints, Hoeffding ignore la variance, et la garantie vaut pour toute distribution (pire cas) alors que la simulation correspond à une configuration particulière. Une borne théorique est une *garantie suffisante*, pas la valeur exacte.

**No-Free-Lunch.** Si les cibles sont complètement aléatoires, les étiquettes des points non vus sont des Bernoulli(1/2) indépendantes de l'échantillon : aucune règle (0, 1, aléatoire, majorité) ne fait mieux que $1/2$ dessus, et le risque moyen vaut $\frac12(1-1/N)^m$, qui ne diminue que par mémorisation. La partie 8 montre aussi que toute règle ayant de bonnes performances pour une famille de cibles en a de mauvaises pour une autre : un apprenant ne peut pas inférer des étiquettes arbitraires non observées à partir de l'échantillon seul.

**Biais inductif.** En supposant que la cible est un seuil et en restreignant $\mathcal H$ aux seuils, on injecte une connaissance a priori qui lie les étiquettes entre elles : un exemple informe sur tout un voisinage. Cette classe est petite ($N-1$ éléments) et contient la cible, donc ERM généralise bien au-delà des points vus (Partie III), avec un risque qui décroît en $\sim 1/m$. La généralisation n'est donc pas gratuite : elle est le produit d'un compromis entre une classe assez riche pour contenir la cible et assez petite pour que la convergence uniforme soit possible avec peu de données.